In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
import numpy as np
import matplotlib.pyplot as plt
from xml.etree import ElementTree
import os
import cv2
import random

In [6]:
class_names = ["trafficlight", "stop", "crosswalk", "speedlimit"]
class_names_label = {class_name: i for i, class_name in enumerate(class_names)}

n_classes = 4
size = (300,400)

In [13]:
import os
import random
import shutil

# Define paths
img_dir = "./images"
label_dir = "./annotations"
train_dir = "./train"
val_dir = "./val"
test_dir = "./test"

# Get file lists and ensure correspondence between images and labels
img_files = sorted(os.listdir(img_dir))  # Sorted for consistent pairing
label_files = sorted(os.listdir(label_dir))  # Ensure matching order

# Check correspondence
assert len(img_files) == len(label_files), "Mismatch between images and labels."

# Pair images and labels
data_pairs = list(zip(img_files, label_files))

# Shuffle the data
random.seed(42)
random.shuffle(data_pairs)

# Split the data
train_size = int(len(data_pairs) * 0.7)
val_size = int(len(data_pairs) * 0.15)

train_data = data_pairs[:train_size]
val_data = data_pairs[train_size:train_size + val_size]
test_data = data_pairs[train_size + val_size:]

# Helper function to create directories
def create_dir_structure(base_dir):
    os.makedirs(os.path.join(base_dir, "images"), exist_ok=True)
    os.makedirs(os.path.join(base_dir, "annotations"), exist_ok=True)

# Create directory structures
for directory in [train_dir, val_dir, test_dir]:
    create_dir_structure(directory)

# Copy files to respective directories
def copy_files(data, dest_dir):
    for img_file, label_file in data:
        shutil.copy(os.path.join(img_dir, img_file), os.path.join(dest_dir, "images", img_file))
        shutil.copy(os.path.join(label_dir, label_file), os.path.join(dest_dir, "annotations", label_file))

copy_files(train_data, train_dir)
copy_files(val_data, val_dir)
copy_files(test_data, test_dir)

print("Data splitting and copying completed.")

Data splitting and copying completed.


In [14]:
def load_data():
    datasets = ['train', 'test', 'val']
    output = []

    for dataset in datasets:
        imags = []
        labels = []
        directoryA = dataset +"/annotations/"
        directoryIMG = dataset +"/images/"
        file = os.listdir(directoryA)
        img = os.listdir(directoryIMG)
        file.sort()
        img.sort()

        i = 0
        for xml in file:

            xmlf = os.path.join(directoryA,xml)
            dom = ElementTree.parse(xmlf)
            vb = dom.findall('object')
            label = vb[0].find('name').text
            labels.append(class_names_label[label])

            img_path = directoryIMG + img[i]
            curr_img = cv2.imread(img_path)
            curr_img = cv2.resize(curr_img, size)
            imags.append(curr_img)
            i +=1
        
        imags = np.array(imags, dtype='float32')
        imags = imags / 255
        
      #  labels = pd.DataFrame(labels)
        labels = np.array(labels, dtype='int32')

        output.append((imags, labels))
    return output


In [15]:
(train_images, train_labels), (test_images, test_labels), (val_images, val_labels) = load_data()

In [29]:
class TSignDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3)
        self.conv2 = nn.Conv2d(32, 64, 3)
        self.conv3 = nn.Conv2d(64, 128, 3)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(215040, 128)
        self.fc2 = nn.Linear(128, 4)

        self.loss_fn = nn.NLLLoss()

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        output = F.log_softmax(x, dim=0)
        return output

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

train_images = torch.tensor(train_images)
train_labels = torch.tensor(train_labels, dtype=torch.long)
train_dataset = TensorDataset(train_images, train_labels)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

In [30]:
def train():
    model = TSignDetector()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    epochs = 10
    for epoch in range(epochs):
        running_loss = 0.0
        for inputs, labels in train_loader:
            optimizer.zero_grad()
            inputs = inputs.permute(0, 3, 1, 2)
            outputs = model(inputs)
            loss = model.loss_fn(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        print(f"Epoch {epoch+1}, Loss: {running_loss}")
    return model

model = train()

Epoch 1, Loss: 105.28688776493073
Epoch 2, Loss: 102.47293210029602
Epoch 3, Loss: 99.49996626377106
Epoch 4, Loss: 96.0581386089325
Epoch 5, Loss: 95.09974730014801
Epoch 6, Loss: 93.80687487125397
Epoch 7, Loss: 90.31542646884918
Epoch 8, Loss: 88.42183065414429
Epoch 9, Loss: 89.92155051231384
Epoch 10, Loss: 88.24618911743164


In [31]:
val_images = torch.tensor(val_images)
val_labels = torch.tensor(val_labels, dtype=torch.long)
val_dataset = TensorDataset(val_images, val_labels)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=True)

test_images = torch.tensor(test_images)
test_labels = torch.tensor(test_labels, dtype=torch.long)
test_dataset = TensorDataset(test_images, test_labels)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)

In [32]:
def get_acc(model, loader):
    total, total_0, total_1, correct, correct_0, correct_1 = 0, 0, 0, 0, 0, 0
    for inputs, labels in loader:
        inputs = inputs.permute(0, 3, 1, 2)
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
        # correct_0 = len([_ for i in range(len(predicted)) if predicted[i] == 0 and labels[i] == 0])
        # correct_1 = len([_ for i in range(len(predicted)) if predicted[i] == 1 and labels[i] == 1])
        # total_0 = len([_ for i in range(len(labels)) if labels[i] == 0])
        # total_1 = len([_ for i in range(len(labels)) if labels[i] == 1])
    print("Total Accuracy", correct/total)
    # print("Class 0 Accuracy", correct_0/total_0)
    # print("Class 1 Accuracy", correct_1/total_1)

print("Validation")
get_acc(model, val_loader)
print("\nTest")
get_acc(model, test_loader)

Validation
Total Accuracy 0.6106870229007634

Test
Total Accuracy 0.5939849624060151
